In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 41. B7 Project — Dynamic Treasury Curve Forecasting Audit

> 外部テストの役割はwinnerを作ることではなく、事前固定した動学modelがrandom walkを超えるかを一度だけ反証することである。

## 学習目標

- B5/B6と同じouter-test境界で5公表日先のcurve forecastを作れる
- random walk、static NS、factor AR、factor VAR、Kalman DNSを比較できる
- maturity別RMSEとDNS coverageを別々に評価できる
- filtered/smoothed、missing、parameter stability、methodology breakを監査できる
- price/hedge/PnL claimをデータ境界から除外できる

## 前提知識

- Week 25–28の全Exit Criteria
- B5/B6のlocked test discipline

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 41


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Locked Project contract

| Field | Value |
|---|---|
| Target | five-tenor curve at (t+5), observed minus predicted in bp |
| Time unit | Treasury publication observations |
| Parameters | fit before B5 outer-test start |
| Online state | filtered only |
| Primary metrics | maturity-level and aggregate RMSE |
| Distribution metric | marginal 90% coverage and width for DNS |
| Secondary horizons | 1 and 20 publication observations |
| Prohibited | test-driven tuning, smoothed-state forecast, bond hedge/PnL claim |

In [4]:
decay = 0.5
loadings = qt.nelson_siegel_loadings(maturity_years, decay)
pretest_mask = curve_dates < test_start_date
pretest_yields = curve_yields[pretest_mask]
pretest_factors = qt.extract_nelson_siegel_factors(pretest_yields, maturity_years, decay)
all_factors = qt.extract_nelson_siegel_factors(curve_yields, maturity_years, decay)

dns = qt.fit_dynamic_nelson_siegel(pretest_yields, maturity_years, decay=decay)
dns_filter = qt.filter_dynamic_nelson_siegel(dns, curve_yields)
factor_var = qt.fit_var(pretest_factors, 1)
factor_ar = [qt.fit_ar(pretest_factors[:, index], 1) for index in range(3)]

stability_rows = []
for label, mask in [
    ("pre-methodology-break", curve_dates < qt.TREASURY_METHOD_BREAK.to_datetime64()),
    ("post-break pretest", (curve_dates >= qt.TREASURY_METHOD_BREAK.to_datetime64()) & pretest_mask),
]:
    if np.sum(mask) >= 30:
        fitted = qt.fit_dynamic_nelson_siegel(curve_yields[mask], maturity_years, decay=decay)
        stability_rows.append(
            {"period": label, "observations": int(np.sum(mask)), "transition_spectral_radius": np.max(np.abs(np.linalg.eigvals(fitted.transition)))}
        )
display(pd.DataFrame(stability_rows))

,period,observations,transition_spectral_radius
0,pre-methodology-break,1733,0.997504
1,post-break pretest,470,0.996961


## 2. One-use outer test

test中も日々の公式curveはforecast originで観察できるためfilter updateとlag historyへ追加できる。一方、transition、variance、decayはpretestで固定する。

In [5]:
primary_horizon = 5
origins = np.flatnonzero(
    (curve_dates >= test_start_date)
    & (np.arange(curve_dates.size) + primary_horizon < curve_dates.size)
)
actual = curve_yields[origins + primary_horizon]
predictions = {
    "random walk": curve_yields[origins],
    "static NS": all_factors[origins] @ loadings.T,
    "factor VAR(1)": np.vstack(
        [qt.forecast_var(factor_var, all_factors[: origin + 1], primary_horizon)[-1] @ loadings.T for origin in origins]
    ),
    "factor AR(1)": np.vstack(
        [
            np.array(
                [qt.forecast_ar(factor_ar[index], all_factors[: origin + 1, index], primary_horizon)[-1] for index in range(3)]
            ) @ loadings.T
            for origin in origins
        ]
    ),
}

dns_predictives = [
    qt.forecast_dynamic_nelson_siegel(
        dns,
        dns_filter.filtered_means[origin],
        dns_filter.filtered_covariances[origin],
        primary_horizon,
    )
    for origin in origins
]
predictions["Kalman DNS"] = np.vstack([item.mean for item in dns_predictives])
dns_standard_deviation = np.vstack([np.sqrt(np.diag(item.covariance)) for item in dns_predictives])

metric_rows = []
for model_name, prediction in predictions.items():
    for tenor_index, tenor in enumerate(qt.DEFAULT_TENORS):
        metric_rows.append(
            {
                "model": model_name,
                "tenor": tenor,
                "rmse_bp": 100.0 * np.sqrt(np.mean((actual[:, tenor_index] - prediction[:, tenor_index]) ** 2)),
                "mae_bp": 100.0 * np.mean(np.abs(actual[:, tenor_index] - prediction[:, tenor_index])),
            }
        )
metric_table = pd.DataFrame(metric_rows)
display(metric_table.pivot(index="model", columns="tenor", values="rmse_bp"))
random_walk_rmse = metric_table.loc[metric_table["model"] == "random walk", "rmse_bp"].to_numpy()
candidate_gate = {}
for model_name in [name for name in predictions if name != "random walk"]:
    candidate_rmse = metric_table.loc[metric_table["model"] == model_name, "rmse_bp"].to_numpy()
    candidate_gate[model_name] = bool(np.all(candidate_rmse < random_walk_rmse))
selected_candidates = [name for name, passed in candidate_gate.items() if passed]
print("candidate gate by model:", candidate_gate)
print("project conclusion:", "no model selected" if not selected_candidates else selected_candidates)

tenor,10y,2y,30y,3m,5y
model,,,,,
Kalman DNS,12.366340,12.487415,11.715332,5.837926,13.260713
factor AR(1),12.543596,12.292412,11.512562,5.112299,13.579730
factor VAR(1),12.493041,12.045785,11.740118,5.704452,13.710970
random walk,12.116059,12.066697,11.581494,5.091053,12.934621
static NS,12.552643,12.062320,11.527190,4.999274,13.662903


candidate gate by model: {'static NS': False, 'factor VAR(1)': False, 'factor AR(1)': False, 'Kalman DNS': False}
project conclusion: no model selected


In [6]:
fig = go.Figure()
for model_name in predictions:
    rows = metric_table[metric_table["model"] == model_name]
    fig.add_scatter(x=rows["tenor"], y=rows["rmse_bp"], name=model_name, mode="lines+markers")
fig.update_layout(
    title="Locked-test five-publication curve RMSE by maturity",
    xaxis_title="Treasury tenor",
    yaxis_title="RMSE (bp)",
    template="plotly_white",
)
fig.show()

## 3. Distribution, horizon, and information-set audits

In [7]:
z90 = 1.6448536269514722
dns_error = actual - predictions["Kalman DNS"]
coverage_rows = []
for tenor_index, tenor in enumerate(qt.DEFAULT_TENORS):
    covered = np.abs(dns_error[:, tenor_index]) <= z90 * dns_standard_deviation[:, tenor_index]
    coverage_rows.append(
        {
            "tenor": tenor,
            "coverage_90": covered.mean(),
            "mean_width_bp": 200.0 * z90 * dns_standard_deviation[:, tenor_index].mean(),
        }
    )
display(pd.DataFrame(coverage_rows))

secondary_rows = []
for horizon in [1, 20]:
    horizon_origins = np.flatnonzero((curve_dates >= test_start_date) & (np.arange(curve_dates.size) + horizon < curve_dates.size))
    horizon_actual = curve_yields[horizon_origins + horizon]
    horizon_dns = np.vstack(
        [
            qt.forecast_dynamic_nelson_siegel(
                dns,
                dns_filter.filtered_means[origin],
                dns_filter.filtered_covariances[origin],
                horizon,
            ).mean
            for origin in horizon_origins
        ]
    )
    secondary_rows.extend(
        [
            {"horizon": horizon, "model": "random walk", "aggregate_rmse_bp": 100.0 * np.sqrt(np.mean((horizon_actual - curve_yields[horizon_origins]) ** 2))},
            {"horizon": horizon, "model": "Kalman DNS", "aggregate_rmse_bp": 100.0 * np.sqrt(np.mean((horizon_actual - horizon_dns) ** 2))},
        ]
    )
display(pd.DataFrame(secondary_rows))

pretest_filter = qt.filter_dynamic_nelson_siegel(dns, pretest_yields)
pretest_smoother = qt.kalman_smoother(pretest_filter, dns.transition)
retrospective_difference = np.mean(
    np.linalg.norm(pretest_filter.filtered_means - pretest_smoother.smoothed_means, axis=1)
)
print("retrospective filtered-smoothed factor difference:", retrospective_difference)
print("forecast inputs are filtered only:", True)

,tenor,coverage_90,mean_width_bp
0,3m,0.952030,24.702253
1,2y,0.911439,41.948027
2,5y,0.913284,44.225849
3,10y,0.909594,41.639653
4,30y,0.922509,40.568157


,horizon,model,aggregate_rmse_bp
0,1,random walk,5.244219
1,1,Kalman DNS,6.014338
2,20,random walk,21.973790
3,20,Kalman DNS,22.720669


retrospective filtered-smoothed factor difference: 0.04357978085366094
forecast inputs are filtered only: True


## 4. Claim audit and unavailable economic evidence

本snapshotにはcoupon cash flows、tradable prices、bid–ask、funding、duration hedge instrumentがない。したがって原カリキュラムのhedge errorは本Core projectでは識別できず、作らない。maturity別yield RMSEは統計的予測精度であり、経済価値ではない。

Project結論は「predeclared modelのhistorical outer-test比較」に限定する。outer testを見た後のwinner採用やdecay再調整は次の新しいholdoutが必要である。

## 5. 失敗モード

- outer testでdecayやstate equationを選ぶ
- full-sample smootherをforecastへ使う
- same-date curve fitをfuture forecastと数える
- marginal 90% intervalをjoint curve coverageと呼ぶ
- yield RMSEをhedge/PnLへ換算する
- 1/20 horizonをprimary結果にすり替える

## 6. 段階別演習

### 基礎

1. model別・tenor別error tableを再現せよ。
2. filteredとsmoothed stateの差をplotせよ。

### 標準

3. validationだけでdecay候補を選ぶ将来protocolを書け。
4. artificial missingnessの率別stress testを追加せよ。

### 研究

5. tradable cash instrumentを合法に取得できる場合のhedge-error estimandを定義せよ。
6. block-aware uncertaintyでRMSE差を評価せよ。

## 7. Exit Criteria

- [ ] B5/B6と同じouter-test開始日を使用した
- [ ] 5公表日先をprimaryとした
- [ ] random walk、static NS、AR、VAR、Kalman DNSを比較した
- [ ] maturity別RMSEとcoverage/widthを分離した
- [ ] forecastにはfiltered stateだけを使った
- [ ] missing、parameter stability、methodology breakを監査した
- [ ] hedge/PnL claimをデータ不足として除外した

## 8. 出典


- [Forecasting: Principles and Practice — Stationarity and differencing](https://otexts.com/fpp3/stationarity.html)
- [Forecasting: Principles and Practice — ARIMA models](https://otexts.com/fpp3/arima.html)
- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)

- [Kalman (1960), A New Approach to Linear Filtering and Prediction Problems](https://people.math.harvard.edu/archive/116_fall_03/handouts/Kalman1960.pdf)
- [Särkkä and Svensson, Bayesian Filtering and Smoothing, 2nd ed.](https://users.aalto.fi/~ssarkka/pub/bfs_book_2023_online.pdf)
- [Diebold and Li, Forecasting the Term Structure of Government Bond Yields](https://www.nber.org/papers/w10048.pdf)